# RO-232 TEB start to finish: every number in the Note

The runs here are **our own Python version of teb_local_planner's algorithm**, written from its code (branch
`ros2-master`, commit e4562a62, the branch Nav2 uses) with its parameter values. It is not the package running.
`images/tebkit.py` copies the code's steps in the code's order; its docstring lists what we simplified (the g2o solver
replaced by our NumPy Levenberg-Marquardt with g2o's step rule, the costmap marked from the room's shapes, the
roadmap's random samples seeded, no computing time). RO-058's band (`images/bandkit.py`, copied) is run again here for
the comparison. All runs are cached in the system's temporary folder; the first run takes a few minutes.

In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
import tebkit as tk, bandkit as bk, carpaths as cp
r2, r3, r4 = (lambda x: round(float(x), 2)), (lambda x: round(float(x), 3)), (lambda x: round(float(x), 4))
R = tk.results()          # the drives with our TEB version
C = tk.car_results()      # the car turning round
B = bk.results()          # RO-058's band, for the comparison
P, F, LEFT = R['person'], R['clear'], R['left']
deg = math.degrees
def obst_of(c):   # the point obstacles of a cycle (stored only for the cycles the animations keep)
    return c['obst'] if c.get('obst') is not None else tk.point_obstacles(tk.costmap(c['x'], c['person']), c['x'])

## 1. Inputs and parameters (Sections 1 and 2)

In [2]:
plan = tk.global_plan()
print('global plan: A* route of RO-034 (', r3(np.sum(np.hypot(*np.diff(bk.ROUTE, axis=0).T))), 'm) with its ends moved to the robot and the door pose:',
      len(plan), 'poses,', r3(tk.plan_length(plan)), 'm, a pose every', r3(tk.plan_length(plan) / (len(plan) - 1)), 'm on average')
print('local costmap:', tk.COSTMAP_SIZE, 'm x', tk.COSTMAP_SIZE, 'm,', tk.COSTMAP_RES, 'm cells:', int(round(tk.COSTMAP_SIZE / tk.COSTMAP_RES)) ** 2, 'cells')
print('controller: every', tk.CYCLE, 's')
print('parameters we changed (ours, C++ default):')
for k, v in tk.OURS.items():
    print('  ', k, v, tk.DEFAULTS[k])
print('box corner on the wheel limit: 0.5 + 1.0 x 0.1 =', 0.5 + 1.0 * 0.1, 'm/s')

global plan: A* route of RO-034 ( 3.414 m) with its ends moved to the robot and the door pose: 75 poses, 3.698 m, a pose every 0.05 m on average
local costmap: 3.0 m x 3.0 m, 0.05 m cells: 3600 cells
controller: every 0.1 s
parameters we changed (ours, C++ default):
   max_global_plan_lookahead_dist 3.0 1.0
   max_vel_x 0.5 0.4
   max_vel_theta 1.0 0.3
   acc_lim_x 0.5 0.5
   acc_lim_theta 10.0 0.5
   min_obstacle_dist 0.1 0.5
   xy_goal_tolerance 0.02 0.2
box corner on the wheel limit: 0.5 + 1.0 x 0.1 = 0.6 m/s


## 2. The plan ahead and the local goal (Section 3)

In [3]:
for k in (0, 24):
    c = P[k]
    x = c['x']
    print('t =', c['t'], 's: robot', np.round(x[:2], 3), 'facing', r2(deg(x[2])), 'deg; pruned plan', len(c['plan']), 'poses; kept', len(c['tplan']),
          'poses; local goal', np.round(c['goal'][:2], 3), 'facing', r2(deg(c['goal'][2])), 'deg,', r4(np.hypot(*(c['goal'][:2] - x[:2]))), 'm from the robot')
print('cut-off distance: 0.85 x', tk.COSTMAP_SIZE / 2, '=', 0.85 * tk.COSTMAP_SIZE / 2, 'm')
c = P[24]; tp = c['tplan']
print('t = 2.4 s: the pose before the local goal is', r4(np.hypot(*(tp[-2, :2] - c['x'][:2]))), 'm away; kept plan length', r3(np.sum(np.hypot(*np.diff(tp[:, :2], axis=0).T))), 'm')

t = 0.0 s: robot [2. 1.] facing 30.0 deg; pruned plan 75 poses; kept 28 poses; local goal [2.75 2.05] facing 90.0 deg, 1.2903 m from the robot
t = 2.4 s: robot [2.387 1.86 ] facing 74.33 deg; pruned plan 75 poses; kept 32 poses; local goal [2.75 3.1 ] facing 90.0 deg, 1.2918 m from the robot
cut-off distance: 0.85 x 1.5 = 1.275 m
t = 2.4 s: the pose before the local goal is 1.2439 m away; kept plan length 1.837 m


In [4]:
# where the local goal's heading jumps and the bands start again
for c in P[44:50]:
    print(c['t'], 'local goal', np.round(c['goal'][:2], 3), 'heading', r2(deg(c['goal'][2])), 'deg', 'restart' if c['planner'].get('reinit') else '')
k = next(i for i, c in enumerate(P) if c.get('planner', {}).get('reinit'))
a, b = P[k - 1]['goal'], P[k]['goal']
print('restart at', P[k]['t'], 's: local goal moved', r3(np.hypot(*(b[:2] - a[:2]))), 'm and turned', r2(abs(deg(tk.wrap(b[2] - a[2])))), 'deg (limits 1.0 m, 90 deg)')
print('restart times: with the person', [c['t'] for c in P if c.get('planner', {}).get('reinit')], ', clear room', [c['t'] for c in F if c.get('planner', {}).get('reinit')])

4.4 local goal [3.25 3.75] heading -35.54 deg 
4.5 local goal [3.328 3.694] heading -35.54 deg 
4.6 local goal [3.406 3.639] heading -35.54 deg 
4.7 local goal [3.483 3.583] heading 90.0 deg restart
4.8 local goal [3.561 3.528] heading 90.0 deg 
4.9 local goal [3.6 3.5] heading 90.0 deg 
restart at 4.7 s: local goal moved 0.096 m and turned 125.54 deg (limits 1.0 m, 90 deg)
restart times: with the person [4.7] , clear room [4.1]


## 3. Obstacles from the costmap (Section 4)

In [5]:
for k in (0, 24):
    c = P[k]
    print('t =', c['t'], 's: lethal cells', c['n_lethal'], ', point obstacles after dropping cells behind the robot', c['n_obst'])
c = P[24]
cells = c['cm']['cells']
print('cells of the person:', int(np.sum(np.hypot(*(cells - bk.PERSON).T) <= bk.PERSON_R + 0.025)))
x = c['x']; d = cells - x[:2]
print('lethal cells behind the robot and more than 1.5 m away:', int(np.sum(((d @ [math.cos(x[2]), math.sin(x[2])]) < 0) & (np.hypot(*d.T) > 1.5))))

t = 0.0 s: lethal cells 272 , point obstacles after dropping cells behind the robot 230
t = 2.4 s: lethal cells 590 , point obstacles after dropping cells behind the robot 534
cells of the person: 60
lethal cells behind the robot and more than 1.5 m away: 26


In [6]:
# one obstacle edge worked by hand: the pose of the picked band closest to the person, at 2.4 s, after optimising
pl = c['planner']; b = pl['candidates'][pl['best']]
idx, O = b.associate(c['obst'])
print('picked band:', len(b.P), 'poses,', len(idx), '(pose, obstacle) pairs; obstacles per pose', np.bincount(idx, minlength=len(b.P)).tolist())
dd = np.hypot(*(b.P[idx, :2] - O).T) - tk.R_ROBOT
j = int(np.argmin(dd)); i = idx[j]
print('closest pair: pose', i, np.round(b.P[i, :2], 4), 'obstacle point', np.round(O[j], 3), 'centre distance', r4(dd[j] + tk.R_ROBOT), 'edge distance d', r4(dd[j]))
e0 = max(0.0, 0.1 + 0.05 - dd[j]); e1 = max(0.0, 0.6 - dd[j])
print('obstacle error', r4(e0), ' inflation error', r4(e1), ' cost with weight 50:', r4(50 * e0 ** 2), ' inflation cost 0.1 x', r4(e1), 'squared =', r4(0.1 * e1 ** 2))

picked band: 24 poses, 28 (pose, obstacle) pairs; obstacles per pose [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 0]
closest pair: pose 10 [2.3417 2.5208] obstacle point [2.725 2.625] centre distance 0.3973 edge distance d 0.1473
obstacle error 0.0027  inflation error 0.4527  cost with weight 50: 0.0004  inflation cost 0.1 x 0.4527 squared = 0.0205


## 4. Warm start and the band from the plan (Section 5)

In [7]:
pl = P[24]['planner']
print('warm-started bands:', [len(w.P) for w in pl['warm']], 'poses; band from the plan:', len(pl['plan_band'].P), 'poses')
pb = pl['plan_band']
print('band from the plan: first time steps', np.round(pb.DT[:3], 4), 's; total', r3(pb.duration()), 's')
q = pb.copy(); q.auto_resize()
print('after resizing:', len(q.P), 'poses, time steps from', r3(q.DT.min()), 'to', r3(q.DT.max()), 's; total', r3(q.duration()), 's')
print('first step: a turn of', r2(deg(abs(tk.wrap(pb.P[1, 2] - pb.P[0, 2])))), 'deg at most 1 rad/s ->', r4(abs(tk.wrap(pb.P[1, 2] - pb.P[0, 2])) / 1.0), 's;',
      r4(np.hypot(*(pb.P[1, :2] - pb.P[0, :2]))), 'm at 0.5 m/s ->', r4(np.hypot(*(pb.P[1, :2] - pb.P[0, :2])) / 0.5), 's')

warm-started bands: [12] poses; band from the plan: 32 poses
band from the plan: first time steps [0.6714 0.101  0.101 ] s; total 4.359 s
after resizing: 17 poses, time steps from 0.2 to 0.4 s; total 4.359 s
first step: a turn of 29.33 deg at most 1 rad/s -> 0.512 s; 0.3357 m at 0.5 m/s -> 0.6714 s


## 5. Homotopy classes (Section 6)

In [8]:
# the H-signature at easy values: a straight line from (-L, d) to (L, d) passing an obstacle at the origin
o = np.array([[0.0, 0.0]])
for d in (0.4, -0.4):
    Pl = np.column_stack([np.linspace(-3, 3, 61), np.full(61, d), np.zeros(61)])
    sweep = math.atan2(d, 3) - math.atan2(d, -3)
    sweep = (sweep + math.pi) % (2 * math.pi) - math.pi
    print('passing', 'above' if d > 0 else 'below', ': angle swept seen from the obstacle', r2(deg(sweep)), 'deg =', r4(sweep), 'rad; sweep / (4 pi) =', r4(sweep / (4 * math.pi)),
          '; H if the 6 m take no time', r4(tk.h_signature(Pl, np.full(60, 1e-11), o)[0]), ', H if they take 12 s', r4(tk.h_signature(Pl, np.full(60, 0.2), o)[0]))

passing above : angle swept seen from the obstacle -164.81 deg = -2.8765 rad; sweep / (4 pi) = -0.2289 ; H if the 6 m take no time -0.2289 , H if they take 12 s -0.4519
passing below : angle swept seen from the obstacle 164.81 deg = 2.8765 rad; sweep / (4 pi) = 0.2289 ; H if the 6 m take no time 0.2289 , H if they take 12 s 0.4519


In [9]:
def side(Pb, c):
    for k in range(len(Pb) - 1):
        y0, y1 = Pb[k, 1], Pb[k + 1, 1]
        if (y0 - c[1]) * (y1 - c[1]) <= 0 and y0 != y1:
            u = (c[1] - y0) / (y1 - y0); xc = Pb[k, 0] + u * (Pb[k + 1, 0] - Pb[k, 0])
            return 'left' if xc < c[0] else 'right'
    return '-'
for name, run, pc in (('person', P, bk.PERSON), ('person 0.25 m further left', LEFT, bk.PERSON_LEFT)):
    for c in run[24:26]:
        pl = c['planner']; rm = pl['roadmap']
        print(name, 't =', c['t'], 's: roadmap', None if rm is None else (len(rm['V']) - 2, 'samples,', len(rm['E']), 'edges,', len(rm['paths']), 'paths,', len(rm['new']), 'new classes'))
        for i, (b, cost) in enumerate(zip(pl['candidates'], pl['costs'])):
            h = pl['classes'][i]; ob = obst_of(c); m = np.hypot(*(ob - pc).T) <= bk.PERSON_R + 0.025
            print('   band', i, side(b.P, pc), 'of the person, H of the person cells', np.round([h[m].min(), h[m].max()], 3),
                  ', cost', r2(cost), '(band from the plan, x 0.95)' if i == pl['init_idx'] else '', '<- picked' if i == pl['best'] else '')

person t = 2.4 s: roadmap (15, 'samples,', 20, 'edges,', 0, 'paths,', 0, 'new classes')
   band 0 left of the person, H of the person cells [-0.447 -0.226] , cost 62.23  
   band 1 left of the person, H of the person cells [-0.459 -0.242] , cost 57.72 (band from the plan, x 0.95) <- picked
person t = 2.5 s: roadmap (15, 'samples,', 19, 'edges,', 4, 'paths,', 0, 'new classes')
   band 0 left of the person, H of the person cells [-0.444 -0.239] , cost 49.62  
   band 1 left of the person, H of the person cells [-0.444 -0.238] , cost 46.74 (band from the plan, x 0.95) <- picked
person 0.25 m further left t = 2.4 s: roadmap (15, 'samples,', 20, 'edges,', 0, 'paths,', 0, 'new classes')
   band 0 left of the person, H of the person cells [-0.616 -0.37 ] , cost 343.59  <- picked
   band 1 right of the person, H of the person cells [0.365 0.607] , cost 361.51 (band from the plan, x 0.95) 
person 0.25 m further left t = 2.5 s: roadmap (15, 'samples,', 20, 'edges,', 4, 'paths,', 1, 'new classes'

## 6. Optimising: the graph and the passes (Section 7)

In [10]:
c = LEFT[25]; pl = c['planner']
for i, hst in enumerate(pl['hist']):
    costs = []
    for k, h in enumerate(hst):
        b = pl['candidates'][i].copy(); b.P, b.DT = h['P'], h['DT']
        costs.append(sum(tk.band_cost(b, c['obst']).values()))
    print('band', i, ': poses', [len(h['P']) for h in hst[::6]], 'at the start of each pass; cost with the set weights', r2(costs[0]), '->', r2(costs[-1]))
b = P[24]['planner']['candidates'][P[24]['planner']['best']]
n = len(b.P)
cnt = dict(time=n - 1, speed_turn=n - 1, acceleration=n - 2 + 2, kinematics=n - 1, obstacle=len(b.associate(P[24]['obst'])[0]))
print('picked band at 2.4 s:', n, 'poses,', n - 1, 'time steps; edges', cnt, 'total', sum(cnt.values()))
print('unknowns:', 3 * (n - 2) + n - 1)

band 0 : poses [21, 20, 21, 21] at the start of each pass; cost with the set weights 31.37 -> 2.42
band 1 : poses [13, 17, 23, 24] at the start of each pass; cost with the set weights 931.43 -> 8.62
band 2 : poses [21, 20, 23, 24] at the start of each pass; cost with the set weights 414.37 -> 18.41
picked band at 2.4 s: 24 poses, 23 time steps; edges {'time': 23, 'speed_turn': 23, 'acceleration': 24, 'kinematics': 23, 'obstacle': 28} total 121
unknowns: 89


## 7. Selecting (Section 8)

In [11]:
for name, run in (('person', P), ('left', LEFT)):
    for c in run[24:26]:
        pl = c['planner']
        print(name, c['t'], 'costs', [r3(v) for v in pl['costs']], 'raw', [r3(b.cost) for b in pl['candidates']], 'plan band', pl['init_idx'], 'picked', pl['best'])

person 2.4 costs [62.234, 57.719] raw [62.234, 60.756] plan band 1 picked 1
person 2.5 costs [49.615, 46.742] raw [49.615, 49.202] plan band 1 picked 1
left 2.4 costs [343.59, 361.514] raw [343.59, 380.541] plan band 1 picked 0
left 2.5 costs [59.023, 314.226, 348.884] raw [59.023, 330.764, 348.884] plan band 1 picked 0


## 8. Check and command (Section 9)

In [12]:
c = P[24]; b = c['planner']['candidates'][c['planner']['best']]
cells = c['cm']['cells']
dmin = [r3(np.min(np.hypot(*(cells - b.P[i, :2]).T)) - tk.R_ROBOT) for i in range(6)]
print('feasibility check at 2.4 s: gap from the disk to the nearest lethal cell at poses 0 to 5:', dmin, '-> feasible', c['feasible'])
gaps = [r3(np.hypot(*(b.P[i + 1, :2] - b.P[i, :2]))) for i in range(5)]
print('distances between those poses', gaps, '(extra poses only above the 0.25 m inscribed radius)')
d0 = np.hypot(*(b.P[1, :2] - b.P[0, :2])); dt0 = b.DT[0]; dth = tk.wrap(b.P[1, 2] - b.P[0, 2])
print('first piece:', r4(d0), 'm and', r4(dth), 'rad in', r4(dt0), 's -> v =', r4(d0 / dt0), 'm/s, omega =', r4(dth / dt0), 'rad/s; command', np.round(c['cmd'], 4))
# a cycle where the band asks for more than 0.5 m/s
for c in P:
    if 'planner' not in c: continue
    b = c['planner']['candidates'][c['planner']['best']]
    v = np.hypot(*(b.P[1, :2] - b.P[0, :2])) / b.DT[0]
    if v > 0.5 + 1e-6:
        print('t =', c['t'], 's: first piece', r4(v), 'm/s -> command', r4(c['cmd'][0]), 'm/s'); break
print('cycles where the band asked for more than 0.5 m/s:', sum(1 for c in P if 'planner' in c and np.hypot(*(c['planner']['candidates'][c['planner']['best']].P[1, :2] - c['planner']['candidates'][c['planner']['best']].P[0, :2])) / c['planner']['candidates'][c['planner']['best']].DT[0] > 0.5 + 1e-6), 'of', sum(1 for c in P if 'planner' in c))
print('infeasible cycles: person', sum(1 for c in P if c.get('feasible') is False), ', left', sum(1 for c in LEFT if c.get('feasible') is False), ', clear', sum(1 for c in F if c.get('feasible') is False))
print('oscillation detected:', sum(1 for c in P if c.get('oscillating')), 'cycles; shorter horizon:', sum(1 for c in P if c.get('shrunk')), 'cycles')

feasibility check at 2.4 s: gap from the disk to the nearest lethal cell at poses 0 to 5: [0.541, 0.475, 0.433, 0.415, 0.427, 0.414] -> feasible True
distances between those poses [0.091, 0.044, 0.018, 0.011, 0.013] (extra poses only above the 0.25 m inscribed radius)
first piece: 0.0906 m and 0.0061 rad in 0.2551 s -> v = 0.3552 m/s, omega = 0.0238 rad/s; command [0.3552 0.0238]
t = 0.7 s: first piece 0.5039 m/s -> command 0.5 m/s
cycles where the band asked for more than 0.5 m/s: 62 of 98
infeasible cycles: person 0 , left 0 , clear 0
oscillation detected: 0 cycles; shorter horizon: 0 cycles


## 9. The whole drive (Section 10)

In [13]:
def summary(run, pc=None):
    xs = np.array([c['x'] for c in run]); ts = np.array([c['t'] for c in run])
    d = np.hypot(*(xs[:, :2] - bk.GOAL[:2]).T); i2 = int(np.argmax(d < 0.02))
    out = dict(arrival=tk.arrival(run), within_2cm=float(ts[i2]), heading_off_deg=r2(deg(abs(tk.wrap(xs[i2, 2] - bk.GOAL[2])))),
               top_speed=r3(max(c['v'] for c in run)), cycles=sum(1 for c in run if 'planner' in c),
               cycles_with_2plus_bands=sum(1 for c in run if 'planner' in c and len(c['planner']['candidates']) > 1),
               most_bands=max(len(c['planner']['candidates']) for c in run if 'planner' in c))
    # Nav2's default goal checker (SimpleGoalChecker, stateful): within 0.25 m, then heading within 0.25 rad
    ixy = int(np.argmax(d < 0.25))
    iyaw = ixy + int(np.argmax(np.abs(tk.wrap(xs[ixy:, 2] - bk.GOAL[2])) < 0.25))
    out.update(nav2_checker=float(ts[iyaw]), nav2_checker_dist=r3(d[iyaw]))
    if pc is not None:
        g = np.hypot(*(xs[:, :2] - pc).T) - bk.PERSON_R - tk.R_ROBOT
        k = int(np.argmin(np.where(ts >= 2.4, g, 9))); out.update(closest=r3(g[k]), closest_at=float(ts[k]))
    return out
print('person:', summary(P, bk.PERSON))
print('clear room:', summary(F))
print('person 0.25 m further left:', summary(LEFT, bk.PERSON_LEFT))
for w in ('1', '50'):
    print('via-points every 0.3 m, weight', w, ':', summary(R['via' + w], bk.PERSON))
L58, F58 = B['loop'], B['free']
x58 = np.array([o['x'] for o in L58])
print('RO-058 band: door at', L58[-1]['t'], 's with the person,', F58[-1]['t'], 's clear; closest', r2(bk.gap(x58[:, 0], x58[:, 1], bk.PERSON).min()), 'm; top speed', r3(max(o['v'] for o in L58)), 'm/s')
print('cost of the person: ours', r2(tk.arrival(P) - tk.arrival(F)), 's, RO-058', r2(L58[-1]['t'] - F58[-1]['t']), 's')

person: {'arrival': 9.8, 'within_2cm': 7.9, 'heading_off_deg': 35.48, 'top_speed': 0.5, 'cycles': 98, 'cycles_with_2plus_bands': 20, 'most_bands': 2, 'nav2_checker': 8.3, 'nav2_checker_dist': 0.04, 'closest': 0.137, 'closest_at': 4.2}
clear room: {'arrival': 7.9, 'within_2cm': 7.0, 'heading_off_deg': 32.38, 'top_speed': 0.5, 'cycles': 79, 'cycles_with_2plus_bands': 0, 'most_bands': 1, 'nav2_checker': 7.4, 'nav2_checker_dist': 0.03}
person 0.25 m further left: {'arrival': 9.8, 'within_2cm': 8.8, 'heading_off_deg': 34.34, 'top_speed': 0.5, 'cycles': 98, 'cycles_with_2plus_bands': 26, 'most_bands': 3, 'nav2_checker': 9.2, 'nav2_checker_dist': 0.031, 'closest': 0.15, 'closest_at': 4.7}
via-points every 0.3 m, weight 1 : {'arrival': 9.3, 'within_2cm': 9.0, 'heading_off_deg': 6.24, 'top_speed': 0.5, 'cycles': 93, 'cycles_with_2plus_bands': 23, 'most_bands': 2, 'nav2_checker': 8.7, 'nav2_checker_dist': 0.055, 'closest': 0.144, 'closest_at': 4.3}
via-points every 0.3 m, weight 50 : {'arrival':

In [14]:
# how far each drive strays from the global plan (mean distance of the robot's path from the plan)
from shapely.geometry import LineString, Point
pl_line = LineString(tk.global_plan()[:, :2])
for name in ('person', 'via1', 'via50'):
    xs = np.array([c['x'] for c in R[name]])
    print(name, 'mean distance from the plan', r3(np.mean([pl_line.distance(Point(p)) for p in xs[:, :2]])), 'm')

person mean distance from the plan 0.176 m
via1 mean distance from the plan 0.153 m
via50 mean distance from the plan 0.118 m


## 10. Car-like robots (Section 11)

In [15]:
rs = cp.reeds_shepp(tuple(tk.CAR_START), tuple(tk.CAR_GOAL), 5.0)
print('Reeds-Shepp, radius 5 m:', cp.word(rs), r3(cp.length(rs)), 'm; 5 pi =', r3(5 * math.pi))
for key, label in (('A', 'straight start, no acceleration limit'), ('B', 'from the Reeds-Shepp path, 0.5 m/s^2')):
    s0, s1 = tk.band_shape(C[key][0]), tk.band_shape(C[key][-1])
    print(label, ': start', {k: r3(v) for k, v in s0.items()}, '-> end', {k: r3(v) for k, v in s1.items()})
b = C['B'][-1]
v = np.hypot(*(b.P[1, :2] - b.P[0, :2])) / b.DT[0]
pr = np.diff(b.P[:2, 0]) * math.cos(b.P[0, 2]) + np.diff(b.P[:2, 1]) * math.sin(b.P[0, 2])
v = float(np.sign(pr[0]) * v); w = float(tk.wrap(b.P[1, 2] - b.P[0, 2]) / b.DT[0])
print('B first piece: v', r4(v), 'omega', r4(w), 'radius', r3(v / w) if w else None, '-> steering', r2(deg(tk.steering_angle(v, w))), 'deg')
print('largest steering angle the conversion allows: atan(2.5 / (0.95 x 5)) =', r2(deg(math.atan(2.5 / 4.75))), 'deg')
paper = {0.75: (4.86, 4.85), 1.75: (5.98, 5.99), 3.0: (9.42, 9.41), 4.25: (13.35, 13.33), 6.75: (21.55, 21.17), 8.0: (24.96, 25.10)}
for rho, s in C['table'].items():
    print('radius', rho, ': ours', r3(s['length']), 'm, Reeds-Shepp', r3(cp.length(cp.reeds_shepp(tuple(tk.CAR_START), tuple(tk.CAR_GOAL), rho))), 'm; paper TEB', paper[rho][0], ', paper RS', paper[rho][1],
          '; reversals', s['reversals'], ', tightest radius', r3(s['r_min']))

Reeds-Shepp, radius 5 m: L+ R- L+ 15.708 m; 5 pi = 15.708
straight start, no acceleration limit : start {'length': 4.0, 'T': 8.0, 'reversals': 1.0, 'r_min': 0.318, 'n': 5.0} -> end {'length': 15.687, 'T': 15.355, 'reversals': 16.0, 'r_min': 4.992, 'n': 90.0}
from the Reeds-Shepp path, 0.5 m/s^2 : start {'length': 15.706, 'T': 15.706, 'reversals': 2.0, 'r_min': 4.999, 'n': 65.0} -> end {'length': 15.68, 'T': 20.546, 'reversals': 2.0, 'r_min': 4.989, 'n': 93.0}
B first piece: v 0.1861 omega 0.0372 radius 4.998 -> steering 26.58 deg
largest steering angle the conversion allows: atan(2.5 / (0.95 x 5)) = 27.76 deg
radius 0.75 : ours 4.836 m, Reeds-Shepp 4.856 m; paper TEB 4.86 , paper RS 4.85 ; reversals 1 , tightest radius 0.681
radius 1.75 : ours 5.982 m, Reeds-Shepp 5.998 m; paper TEB 5.98 , paper RS 5.99 ; reversals 1 , tightest radius 1.729
radius 3.0 : ours 9.397 m, Reeds-Shepp 9.425 m; paper TEB 9.42 , paper RS 9.41 ; reversals 10 , tightest radius 2.988
radius 4.25 : ours 13.332 m, 

## 11. The hand calculations in the Note, with enough digits

In [16]:
r5 = lambda x: round(float(x), 5)
c = P[24]; b = c['planner']['candidates'][c['planner']['best']]
d0 = np.hypot(*(b.P[1, :2] - b.P[0, :2])); dt0 = b.DT[0]; dth = tk.wrap(b.P[1, 2] - b.P[0, 2])
print('command at 2.4 s: distance', r5(d0), 'm, time', r5(dt0), 's, turn', r5(dth), 'rad -> v', r5(d0 / dt0), ', omega', r5(dth / dt0))
pb = c['planner']['plan_band']
print('plan band first piece: distance', r5(np.hypot(*(pb.P[1, :2] - pb.P[0, :2]))), 'm, turn', r5(abs(tk.wrap(pb.P[1, 2] - pb.P[0, 2]))), 'rad, dt', r5(pb.DT[0]))
bb = C['B'][-1]
v = np.hypot(*(bb.P[1, :2] - bb.P[0, :2])) / bb.DT[0]; w = tk.wrap(bb.P[1, 2] - bb.P[0, 2]) / bb.DT[0]
print('car B first piece: v', r5(v), 'omega', r5(w), 'radius', r5(v / w), 'steering', r5(deg(math.atan(2.5 / (v / w)))), 'deg')
for name, run in (('person', P), ('left', LEFT)):
    for cc in run[24:26]:
        print(name, cc['t'], 'raw costs', [r5(x.cost) for x in cc['planner']['candidates']], 'selection', [r5(x) for x in cc['planner']['costs']])
c7 = P[7]; b7 = c7['planner']['candidates'][c7['planner']['best']]
print('t = 0.7 s: band first piece', r5(np.hypot(*(b7.P[1, :2] - b7.P[0, :2]))), 'm in', r5(b7.DT[0]), 's ->', r5(np.hypot(*(b7.P[1, :2] - b7.P[0, :2])) / b7.DT[0]), 'm/s; command', np.round(c7['cmd'], 5))

command at 2.4 s: distance 0.09061 m, time 0.25511 s, turn 0.00607 rad -> v 0.35516 , omega 0.02378
plan band first piece: distance 0.33572 m, turn 0.51196 rad, dt 0.67144
car B first piece: v 0.18613 omega 0.03724 radius 4.99761 steering 26.576 deg
person 2.4 raw costs [62.23364, 60.75637] selection [62.23364, 57.71856]
person 2.5 raw costs [49.61509, 49.20182] selection [49.61509, 46.74173]
left 2.4 raw costs [343.59028, 380.54116] selection [343.59028, 361.5141]
left 2.5 raw costs [59.0227, 330.7645, 348.88395] selection [59.0227, 314.22627, 348.88395]
t = 0.7 s: band first piece 0.12509 m in 0.24825 s -> 0.5039 m/s; command [0.5     0.19286]


## 12. Checks behind statements in the text

In [17]:
for name, run in (('person', P), ('clear', F), ('left', LEFT)):
    xs = np.array([c['x'] for c in run])
    print(name, ': infeasible', sum(1 for c in run if c.get('feasible') is False), ', oscillation', sum(1 for c in run if c.get('oscillating')),
          ', shorter horizon', sum(1 for c in run if c.get('shrunk')), '; path driven', r3(np.sum(np.hypot(*np.diff(xs[:, :2], axis=0).T))), 'm')
print('end of the drive with the person: t, poses in the picked band, command')
for c in P[79:98:2]:
    b = c['planner']['candidates'][c['planner']['best']]
    print('  ', c['t'], len(b.P), np.round(c['cmd'], 3), 'heading off', r2(deg(abs(tk.wrap(c['x'][2] - bk.GOAL[2])))), 'deg')
g = P[24]['goal']
print('local goal at 2.4 s is', r3(np.hypot(*(g[:2] - bk.PERSON)) - bk.PERSON_R), 'm from the edge of the person')
g47 = P[47]['goal']
print('local goal at 4.7 s:', np.round(g47[:2], 3), r3(np.hypot(*(g47[:2] - bk.GOAL[:2]))), 'm from the door pose')
c = LEFT[25]; pl = c['planner']; rm = pl['roadmap']; ob = c['obst']
new_b = rm['new'][0][1]; hn = tk.h_signature(new_b.P, new_b.DT, ob); hp = tk.h_signature(pl['plan_band'].P, pl['plan_band'].DT, ob)
hl = tk.h_signature(pl['warm'][0].P, pl['warm'][0].DT, ob)
for path, verdict in rm['verdicts']:
    bb = tk.Band.from_waypoints([rm['V'][k] for k in path], c['tplan'][0][2], c['tplan'][-1][2])
    h = tk.h_signature(bb.P, bb.DT, ob)
    print('path', path, verdict, ': same class as the left band', tk.same_class(h, hl), ', the plan band', tk.same_class(h, hp), ', the new band', tk.same_class(h, hn))

person : infeasible 0 , oscillation 0 , shorter horizon 0 ; path driven 3.709 m
clear : infeasible 0 , oscillation 0 , shorter horizon 0 ; path driven 3.197 m
left : infeasible 0 , oscillation 0 , shorter horizon 0 ; path driven 3.972 m
end of the drive with the person: t, poses in the picked band, command
   7.9 4 [0.128 0.973] heading off 35.48 deg
   8.1 3 [-0.015  0.985] heading off 24.17 deg
   8.3 3 [-0.147  0.946] heading off 13.05 deg
   8.5 3 [-0.173  0.119] heading off 6.59 deg
   8.7 3 [-0.041 -0.087] heading off 5.57 deg
   8.9 3 [0.075 0.083] heading off 5.84 deg
   9.1 3 [0.111 0.213] heading off 5.22 deg
   9.3 3 [0.008 0.074] heading off 4.16 deg
   9.5 3 [-0.112  0.228] heading off 1.55 deg
   9.7 3 [ 0.014 -0.067] heading off 2.21 deg
local goal at 2.4 s is 0.181 m from the edge of the person
local goal at 4.7 s: [3.483 3.583] 0.143 m from the door pose
path [0, 1, 16] taken : same class as the left band True , the plan band False , the new band False
path [0, 5, 10, 